# CNN Fundamentals using PyTorch

### Import and verify

In [ ]:
import torch

print(torch.__version__)

2.11.0+cpu


### Image as Tensors

**1. The Digital Grid (Spatial Dimensions)**
*   **Concept:** A computer does not "see" shapes or lighting; it only reads discrete numerical values arranged in a rigid 2D grid. Each discrete point in this grid is a pixel (picture element).
*   **Mathematics:** If an image is $H$ pixels high and $W$ pixels wide, the spatial resolution is mathematically represented as a matrix $M \in \mathbb{R}^{H \times W}$.

**2. The Depth Dimension (Color Channels)**
*   **Concept:** To represent color, the computer stacks multiple spatial matrices on top of each other. These are called **Channels**.
    *   A grayscale image has 1 channel (representing intensity from black to white).
    *   A standard digital color image uses the RGB color model, meaning it is composed of 3 distinct channels: Red, Green, and Blue.
*   **Mathematics:** The complete color image is a Rank-3 Tensor $T \in \mathbb{R}^{C \times H \times W}$. A $256 \times 256$ pixel RGB image is mathematically a tensor of shape `(3, 256, 256)`.

**3. Pixel Value Storage (Data Types)**
*   **Concept:** In standard file formats (like JPEG or PNG), the value of a single color channel for a single pixel is stored as an 8-bit unsigned integer (`uint8`).
*   **Mathematics:** An 8-bit integer can hold $2^8 = 256$ discrete values. Therefore, the numerical domain of raw image data is strictly bounded: $x \in [0, 255]$.
    *   $0$ represents the absolute absence of that color (black).
    *   $255$ represents the maximum intensity of that color.

**4. The Mathematics of Image Normalization**
*   **The CS Problem:** Standard neural networks rely on gradient descent. If we feed a matrix containing values up to $255$ into a network initialized with tiny weights (e.g., $0.01$), the dot products will produce massive numbers. This causes gradients to destabilize (explode) during the backward pass, mathematically preventing the network from converging.
*   **The Solution (Min-Max Scaling):** We must compress the $0-255$ integer range into a floating-point range of $0.0$ to $1.0$.
    *   **Formula:** $x_{scaled} = \frac{x_{raw}}{255.0}$
*   **The Solution (Standardization/Z-Score Normalization):** To further optimize hardware computation, computer scientists go a step further. They calculate the statistical Mean ($\mu$) and Standard Deviation ($\sigma$) of the entire image dataset, and shift the tensor values so the data is centered around zero with a standard deviation of 1.
    *   **Formula:** $x_{norm} = \frac{x_{scaled} - \mu}{\sigma}$

**5. The GPU Batch Format (B, C, H, W)**
*   **Concept:** As we learned in the previous module, GPUs are designed for massive parallel throughput. We never send a single `(C, H, W)` image to the GPU. We stack $B$ number of images into a contiguous block of memory.
*   **Mathematics:** The final Rank-4 Tensor sent to the hardware is $T_{batch} \in \mathbb{R}^{B \times C \times H \times W}$.

In [ ]:
""" IMAGE LOADING, TRANSFORMATION AND EXECUTION PIPELINE """

import torch
import torchvision.transforms as transforms
import numpy as np
from PIL import Image

print(torch.__version__)
print(np.__version__)
print(Image.__version__)

2.11.0+cu128
2.0.2
11.3.0


In [ ]:
# Set the GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Target Hardware: {device}")

Target Hardware: cuda


In [ ]:
# Simulating reading a raw image (.jpg) from SSD / HDD

# Generating a random 256x256 (H x W) image with random pixel values between 0 - 255
# Current shape in Height x Width x Channels and the type is 8 bit unsigned integer
raw_memory_block = np.random.randint(0, 256, size=(256, 256, 3), dtype=np.uint8)
raw_image = Image.fromarray(raw_memory_block)

print("INITIAL STATE (RAM)")
print(raw_memory_block[10:15, 10:15])
print(raw_image)
print("Data Type (raw memory block): ", type(raw_memory_block))
print("Data Type: ", type(raw_image))
print("Raw Shape (HWC): ", raw_memory_block.shape)

INITIAL STATE (RAM)
[[[ 82 153 114]
  [208  26 162]
  [126 178  12]
  [ 92 136  99]
  [ 26 205  47]]

 [[165 239  82]
  [119 245  80]
  [152 102 224]
  [ 17 161  32]
  [175   9 131]]

 [[226 250  96]
  [ 76 143 220]
  [112 101 111]
  [ 12 100 176]
  [138  57 214]]

 [[216  36 243]
  [123  35  33]
  [101 104 207]
  [ 89  77  87]
  [125  35  45]]

 [[228 110 168]
  [142 129  30]
  [ 80 121  26]
  [200 201 240]
  [ 90 244 103]]]
<PIL.Image.Image image mode=RGB size=256x256 at 0x79982674CD10>
Data Type (raw memory block):  <class 'numpy.ndarray'>
Data Type:  <class 'PIL.Image.Image'>
Raw Shape (HWC):  (256, 256, 3)


#### Image Transformation Pipeline Breakdown

The image pipeline sequentially converts a **raw PIL Image or NumPy array** into a **normalized PyTorch tensor** with a reshaped memory layout, making it ready for deep learning models.

##### 1. Input Data Format
* **Format**: A raw RGB image loaded via a library like PIL or NumPy.
* **Shape**: $H \times W \times C$ (Height, Width, Channels).
* **Data Range**: Integer values from $0$ to $255$.

##### 2. Step-by-Step Execution

###### 1. Convert to Tensor
* **Action**: `transforms.ToTensor()` processes the raw image.
* **Layout Shift**: Rearranges the dimensions from $H \times W \times C$ to $C \times H \times W$ (Channels, Height, Width) to match PyTorch expectations.
* **Value Scaling**: Scales all pixel values down from integers to a floating-point range of $[0.0, 1.0]$.

###### 2. Standardize Values
* **Action**: `transforms.Normalize()` applies a statistical shift to each color channel independently.
* **Formula**: It computes the new value for each pixel using the formula:
$$\text{Output} = \frac{\text{Pixel Value} - \text{Mean}}{\text{Std}}$$
* **Data Shift**: It shifts the pixel values from $[0.0, 1.0]$ to a range centered around zero (typically between roughly $-2.1$ and $2.6$), matching the distribution of the ImageNet dataset.

###### 3. Output Data Format
* **Format**: A `torch.Tensor` object ready for GPU processing.
* **Shape**: $C \times H \times W$ ($3$ channels $\times$ Height $\times$ Width).
* **Data Range**: Normalized floating-point numbers centered around $0.0$.

The raw image passes through structural rearrangement and mathematical scaling to produce an optimized, model-ready tensor.


In [ ]:
# Image transformation pipeline

# Standard deep learning vision pipeline
# Declaraing mathematical opeations the CPU will perform on the image before sending it to the GPU
# Input data format: Memory Layout (Shape) = HWC, Format = RGB, Data Range = 0 to 255 per pixel per layer.
image_pipeline = transforms.Compose([
    # First operation: trasnforms.ToTensor()
    # Convert the image into tensor
    # Changes the memory layout from HWC (256, 256, 3) to CHW (3, 256, 256)
    # Casts the uint8 integers to float32 and divides every pixel by 255.0
    transforms.ToTensor(),
    # Second operation: transforms.Normalize()
    # Execute the statistical shift
    # Using the mean and standard deviation numbers from ImageNet dataset (universal standard)
    transforms.Normalize(mean=[0.485, 0.456, 0.406],    # Mean for R, G, B
                         std=[0.299, 0.244, 0.255]      # Standard Deviation for R, G, B
    )
])
# Output data format: torch.Tensor() ready for GPU, Memory Layout (Shape) = CHW, Data Range = Normalized floating-point numbers centered around 0.0

In [ ]:
# Exuting the image transformation pipeline and loading it onto the GPU

print("Executing Image Transformation Pipeline")

# Run the image through the image transform pipeline
processed_tensor = image_pipeline(raw_image)
print("Updated Data Type:", processed_tensor.dtype)
print("Updated Shape:", processed_tensor.shape)
print(f"Min Max Range of data: Min={processed_tensor.min():.2f}, Max={processed_tensor.max():.2f}")

# Construct image batch (video) for GPU loading
batch_tensor = processed_tensor.unsqueeze(0)
print(f"Batch Shape (B, C, H, W):", batch_tensor.shape)

# Load the batch to the GPU through PCIE
gpu_batch = batch_tensor.to(device)
print("Final State (VRAM)")
print("Device:", gpu_batch.device)

# The image is now in GPU VRAM and ready to servce for CNN training

Executing Image Transformation Pipeline
Updated Data Type: torch.float32
Updated Shape: torch.Size([3, 256, 256])
Min Max Range of data: Min=-1.87, Max=2.33
Batch Shape (B, C, H, W): torch.Size([1, 3, 256, 256])
Final State (VRAM)
Device: cuda:0


### Convolution

#### Convolution Operation (Edge Detection) in Image

**1. The Input Tensor (The Image)**
We load a small $6 \times 6$ pixel region of the road into RAM.
The left half is bright white (pixel value 10), and the right half is dark asphalt (pixel value 0).
<br><br>
$$
\text{Input } (I) =
\begin{bmatrix}
10 & 10 & 10 & 0 & 0 & 0 \\
10 & 10 & 10 & 0 & 0 & 0 \\
10 & 10 & 10 & 0 & 0 & 0 \\
10 & 10 & 10 & 0 & 0 & 0 \\
10 & 10 & 10 & 0 & 0 & 0 \\
10 & 10 & 10 & 0 & 0 & 0
\end{bmatrix}
$$
<br>
**2. The Learnable Weights (The Kernel)**
Instead of random numbers, let's assume our CNN has already "learned" the exact weights required to detect a vertical edge. This specific $3 \times 3$ matrix is known mathematically as the Prewitt vertical operator.
<br><br>
$$
\text{Kernel } (K) =
\begin{bmatrix}
 1 & 0 & -1 \\
 1 & 0 & -1 \\
 1 & 0 & -1
\end{bmatrix}
$$
<br>
**3. The Execution: Scanning a Flat Region**
The GPU places the $3 \times 3$ kernel over the top-left corner of the input image. This region is entirely bright white `(10)`.
The CPU/GPU computes the element-wise multiplication and sums the result (the dot product):
Using Convolution Formula:
<br>
$$S(i, j) = \sum_{m} \sum_{n} I(i + m, j + n) \cdot K(m, n)$$
<br>
- **The Goal:** We want to calculate a single number for our output Feature Map ($S$) at the specific 2D coordinate $(i, j)$.
- **The** $\Sigma \Sigma$ **(The 2D Loop):** The double Sigma is just a 2D nested `for` loop in computer science. It iterates over the height ($m$) and width ($n$) of your tiny $3 \times 3$ Kernel ($K$).
- $I(i + m, j + n)$ **(The Offset):** $I$ is your massive input image. $(i, j)$ is the current anchor point we are looking at. $+m$ and $+n$ act as spatial offsets. The math is saying: *"Lock onto coordinate* $(i,j)$ *in the main image. Now, read the small* $3 \times 3$ *grid of memory addresses directly surrounding it."*
- $\cdot K(m, n)$ **(The Dot Product):** We take the value extracted from the image and multiply it by the specific weight located at $(m, n)$ inside the Kernel. The $\Sigma$ commands us to add all 9 of these multiplication results together into one final number.
<br>
$$
\text{Patch} =
\begin{bmatrix}
10 & 10 & 10 \\
10 & 10 & 10 \\
10 & 10 & 10
\end{bmatrix}
*
\begin{bmatrix}
 1 & 0 & -1 \\
 1 & 0 & -1 \\
 1 & 0 & -1
\end{bmatrix}
$$
<br>
$$
\text{Calculation: } (10\times1 + 10\times0 + 10\times-1) \times 3 \text{ rows}
$$
<br>
$$
\text{Result: } (10 + 0 - 10) + (10 + 0 - 10) + (10 + 0 - 10) = \mathbf{0}
$$
<br>
*CS Conclusion:* The output is 0. The kernel mathematically proves there is no vertical edge in this specific patch of memory.

**4. The Execution: Hitting the Edge**
The GPU strides the kernel over to the middle of the image, where the 10s meet the 0s.
<br><br>
$$
\text{Patch} =
\begin{bmatrix}
10 & 10 & 0 \\
10 & 10 & 0 \\
10 & 10 & 0
\end{bmatrix}
*
\begin{bmatrix}
 1 & 0 & -1 \\
 1 & 0 & -1 \\
 1 & 0 & -1
\end{bmatrix}
$$
<br>
$$
\text{Calculation: } (10\times1 + 10\times0 + 0\times-1) \times 3 \text{ rows}
$$
<br>
$$
\text{Result: } (10 + 0 + 0) + (10 + 0 + 0) + (10 + 0 + 0) = \mathbf{30}
$$
<br>
*CS Conclusion:* The output is a massive positive number (30). The kernel has successfully "activated." This high numerical value tells the next layer of the neural network: *"I found a stark vertical boundary at this exact spatial coordinate."*

**5. The Output Feature Map**
After sliding across the entire $6 \times 6$ image (with a stride of 1 and no padding), the resulting Feature Map is a $4 \times 4$ tensor.

$$
\text{Feature Map } =
\begin{bmatrix}
0 & 30 & 30 & 0 \\
0 & 30 & 30 & 0 \\
0 & 30 & 30 & 0 \\
0 & 30 & 30 & 0
\end{bmatrix}
$$
<br>
The network has successfully transformed a grid of raw pixel intensities into a mathematical map of geometric features. If we configure a convolutional layer with 64 `out_channels`, the GPU runs 64 of these distinct filters simultaneously, extracting horizontal edges, diagonal lines, and color blobs all in one massive parallel operation.

In [ ]:
""" SIMULATING THE CONVOLUTION OPERATION """

import torch
import torch.nn as nn

In [ ]:
# Harware setup, sample data allocation then moving the sample data to GPU

# Set the device as GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

# Allocate a sample batch of images in VRAM
# Batch Size (No. of images) (B) = 32, RGB Channels (C) = 3, Height (H) = 64, Width (W) = 64
batch_size = 32
input_tensor = torch.randn(size=(batch_size, 3, 64, 64)).to(device)
print(f"Input tensor shape: {input_tensor.shape}")
print(f"Input tensor device: {input_tensor.device}")

cuda
Input tensor shape: torch.Size([32, 3, 64, 64])
Input tensor device: cuda:0


In [ ]:
# Defining a class for the Convolution operation and creating an object of it

class ConvOperation(nn.Module):
    def __init__(self, stride_jump):
        super().__init__()

        self.conv_layer = nn.Conv2d(
            in_channels=3,              # The depth of the input tensor (image color channels) (3 for R, G, B)
            out_channels=16,            # How many feature maps we want to generate to extract distinct viusal patterns from our RGB images.
            kernel_size=3,              # 3x3 sliding window (filter) (GPU will create 16 separate 3x3 kernels)
            stride=stride_jump,         # The window jumps certain pixel (eg: 1 pixel or 2 pixels) at a time to the right or down (if started form top left)
            padding=1                   # We add a 1 pixel border of 0s to preserve spatial dimentions
        )

        self.conv_layer = self.conv_layer.to(device)

    # The forward pass (prediction)
    def forward(self, input_tensor):
        output = self.conv_layer(input_tensor)
        return output

# Instantiate an object for the convolution operation with default random weights and stride jump rate of 1
conv_model = ConvOperation(1)
# Shape: [16, 3, 3, 3]
# 16 sets of kernels to produce 16 feature maps on convolution operation
# 3 kernels for each input channel (R, G, B)
# Each kernel has H=3, W=3
print(f"Allocated kernel weights shape: {conv_model.conv_layer.weight.shape}")
print(f"Peeking into a sample of the weights:\n{conv_model.conv_layer.weight[5:6, 1:2, :, :]}")

Allocated kernel weights shape: torch.Size([16, 3, 3, 3])
Peeking into a sample of the weights:
tensor([[[[-0.1101, -0.0068,  0.0830],
          [-0.0056, -0.1208, -0.0960],
          [ 0.1373, -0.1130, -0.1726]]]], device='cuda:0',
       grad_fn=<SliceBackward0>)


In [ ]:
# Execution of the convolution operation using our input tensor and mathematical verification

# The forward pass (extracting the feature maps)
# The feature maps should of 64 x 64
output_tensor = conv_model(input_tensor)
# Shpae: [32, 16, 64, 64]
# 32 sets of feature maps for 32 images
# 16 feature maps in each set
# Each feature maps has H=64, W=64
print(output_tensor.shape)
print(output_tensor.device)

torch.Size([32, 16, 64, 64])
cuda:0


Convolution Output Shape Forumula (precalculating the Width of each output feature map):

$$W_{out} = \lfloor \frac{W_{in} - K + 2P}{S} \rfloor + 1$$
<br>
- $W_{in}$**:** Width of the input image.
- $W_{out}$**:** Width of the output feature map.
- $W_{in} + 2P$**:** This is the total physical memory footprint. If the input width is $5$ and Padding is $1$, we add $1$ column of zeros to the left and $1$ to the right. Total physical width is $5 + 2(1) = mathbf{7}$.
- **$- K$:** Why subtract the Kernel size? Because the Kernel cannot slide off the edge of the memory block. If you have a $3$-pixel wide kernel sliding across $7$ pixels, the very first placement consumes pixels index $0, 1, 2$. It cannot start at index $6$ because there is no data at index $7$ or $8$. We subtract $K$ to find the total *slidable distance*.
- **$/ S$:** We divide the slidable distance by the Stride. If Stride is $2$, we skip every other pixel, literally cutting the number of operations (and the output dimension) in half.
- **$+ 1$:** We must add $1$ at the very end to account for the very first valid placement of the kernel at index $(0,0)$ before any sliding movement actually occurred.

We use the same formula to calculate the height as well:
1.  **Vertical Calculation:** $H_{out} = \left\lfloor \frac{H_{in} - K_h + 2P_h}{S_h} \right\rfloor + 1$
2.  **Horizontal Calculation:** $W_{out} = \left\lfloor \frac{W_{in} - K_w + 2P_w}{S_w} \right\rfloor + 1$

In [ ]:
# Verification of the output tensor shape using the Convolution Output Shape formula

W_in = input_tensor.shape[3]                # Width of input images
K = conv_model.conv_layer.weight.shape[3]   # Kernel width
P = conv_model.conv_layer.padding[1]        # No. of padding
S = conv_model.conv_layer.stride[1]         # Stride jump length

print("For each image, each channel:")
print(f"Input width: {W_in}\nKernel width: {K}\nNo. of padding: {P}\nStride jump length: {S}")
W_out = int(((W_in - K + 2 * P) / S) + 1)
print(f"Width and potential height of the output feature map: {W_out}")

For each image, each channel:
Input width: 64
Kernel width: 3
No. of padding: 1
Stride jump length: 1
Width and potential height of the output feature map: 64


In [ ]:
# Altering the stride jump length to 2 to reduce memory usage

downsampler_model = ConvOperation(2)

# Executing the forward pass and extracting the feature maps
downsampler_output = downsampler_model(input_tensor)
print(downsampler_output.shape)
print(downsampler_output.device)

# Verification of the downsampler output tensor shape using the Convolution Output Shape formula

W_in1 = input_tensor.shape[3]                           # Width of input images
H_in1 = input_tensor.shape[2]                           # Height of input images
Kw1 = downsampler_model.conv_layer.weight.shape[3]      # Kernel width
Kh1 = downsampler_model.conv_layer.weight.shape[2]      # Kernal height
P1 = downsampler_model.conv_layer.padding[1]            # No. of padding
S1 = downsampler_model.conv_layer.stride[1]             # Stride jump length

print("For each image, each channel:")
print(f"Input width: {W_in1}\nInput height: {H_in1}\nKernel width: {Kw1}\nKernel Height: {Kh1}\nNo. of padding: {P1}\nStride jump length: {S1}")
W_out1 = int(((W_in1 - Kw1 + 2 * P1) / S1) + 1)
H_out1 = int(((H_in1 - Kh1 + 2 * P1) / S1) + 1)
print(f"Width of the output feature map: {W_out1}")
print(f"Height of the output feature map: {H_out1}")

# Step by step calculation of the Convolution Output Shape Formula
# W_out = floor((64 - 3 + 2) / 2) + 1
# W_out = floor(63 / 2) + 1
# W_out = floor(31.5) + 1
# W_out = 31 + 1 = 32

torch.Size([32, 16, 32, 32])
cuda:0
For each image, each channel:
Input width: 64
Input height: 64
Kernel width: 3
Kernel Height: 3
No. of padding: 1
Stride jump length: 2
Width of the output feature map: 32
Height of the output feature map: 32


### Pooling & Hierarchies

**1. The Pooling Operation (Non-Learnable Downsampling)**
*   **Concept:** Pooling uses a sliding window over the **Feature map** exactly like Convolution. However, instead of computing a dot product with learnable weights, it executes a fixed, non-learnable mathematical reduction operator and outputs a separate tensor.
*   **Max Pooling:** The most common variant. As the window slides, it simply outputs the maximum numerical value found within that spatial window.
*   **Average (Mean) Pooling:** Computes the mathematical mean of the values in the window.

**2. The Mathematics of Max Pooling**
*   **The Algorithm:** Given a feature map $F$ and a $2 \times 2$ pooling window $W$, the calculation for the output region is: $P_{out} = \max_{(i,j) \in W} F(i, j)$.
*   **Stride and Overlap:** In pooling, the Stride ($S$) is traditionally set to equal the Kernel size ($K$). If $K=2$, then $S=2$. This means the windows *do not overlap*.
*   **Spatial Reduction:** Using the Master Formula $W_{out} = \lfloor \frac{W_{in} - K}{S} \rfloor + 1$:
    *   If $W_{in} = 64$, $K = 2$, $S = 2$.
    *   $W_{out} = \lfloor \frac{64 - 2}{2} \rfloor + 1 = 31 + 1 = 32$.
    *   A $2 \times 2$ Max Pool with a Stride of 2 mathematically cuts the spatial height and width exactly in half, discarding 75% of the total tensor volume in a single operation.

#### Demo of Pooling
Let's look at a strict mathematical scenario demonstrating exactly how Max Pooling solves the "Spatial Hyper-Sensitivity" problem. We call this mathematical property **Translation Invariance**.

**The Scenario: The Jittering Camera**
Imagine our self-driving car's camera is vibrating slightly. We pass two consecutive frames through our Convolutional layer. The Conv layer successfully detects a specific feature (like a stop sign's edge) and outputs a high activation number (let's use `9`).

Because the camera vibrated, the spatial location of the `9` shifts by one pixel in memory between Frame 1 and Frame 2.

**3. Frame 1 Feature Map (Before Pooling)**
The edge is detected in the absolute top-left memory address `(0, 0)`.

$$
\text{Feature Map of Frame 1} =
\begin{bmatrix}
\mathbf{9} & 1 & 0 & 2 \\
3 & 4 & 1 & 0 \\
0 & 2 & 5 & 1 \\
1 & 0 & 2 & 3
\end{bmatrix}
$$

**4. Frame 2 Feature Map (Before Pooling - The Shift)**
The camera vibrates. The exact same edge is detected, but its mathematical coordinate has shifted diagonally to `(1, 1)`. To a rigid matrix multiplication layer, this is a completely different input array.

$$
\text{Feature Map of Frame 2} =
\begin{bmatrix}
1 & 3 & 0 & 2 \\
4 & \mathbf{9} & 1 & 0 \\
0 & 2 & 5 & 1 \\
1 & 0 & 2 & 3
\end{bmatrix}
$$

**3. The Max Pooling Execution (Stride = 2, Kernel = 2)**
The GPU allocates a new, smaller tensor. It divides the $4 \times 4$ Feature Map into four distinct $2 \times 2$ quadrants and extracts only the maximum value from each.

**Processing Frame 1:**
*   Top-Left Quadrant: $\max(9, 1, 3, 4) = \mathbf{9}$
*   Top-Right Quadrant: $\max(0, 2, 1, 0) = \mathbf{2}$
*   Bottom-Left Quadrant: $\max(0, 2, 1, 0) = \mathbf{2}$
*   Bottom-Right Quadrant: $\max(5, 1, 2, 3) = \mathbf{5}$

$$
\text{Pooled Output 1} =
\begin{bmatrix}
\mathbf{9} & 2 \\
2 & 5
\end{bmatrix}
$$

**Processing Frame 2:**
*   Top-Left Quadrant: $\max(1, 3, 4, 9) = \mathbf{9}$
*   Top-Right Quadrant: $\max(0, 2, 1, 0) = \mathbf{2}$
*   Bottom-Left Quadrant: $\max(0, 2, 1, 0) = \mathbf{2}$
*   Bottom-Right Quadrant: $\max(5, 1, 2, 3) = \mathbf{5}$

$$
\text{Pooled Output 2} =
\begin{bmatrix}
\mathbf{9} & 2 \\
2 & 5
\end{bmatrix}
$$

##### The Conclusion
Look at the two output matrices. **They are mathematically identical.**

Even though the raw data moved to different memory addresses in the upstream tensors, the Pooling layer aggressively filtered the data, absorbing the spatial shift.

The next Convolutional layer deeper in the network will receive the exact same `2x2` input tensor in both scenarios. The network's mathematical behavior remains perfectly stable. It learned that the feature *exists in the top-left region*, without overfitting to the exact pixel coordinate.

Furthermore, any feature map passing through this operation drops 75% of its memory footprint (from 16 numbers down to 4), allowing the GPU to process drastically deeper architectural hierarchies without running out of VRAM.

In [ ]:
""" SIMULATING THE POOLING OPERATION """

import torch
import torch.nn as nn

In [ ]:
# GPU setup

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [ ]:
# Frames setup for the simulation

# Feature map of frame 1 with single channel
f_map_1 = torch.tensor([[[
    [9.0, 1.0, 0.0, 2.0],
    [3.0, 4.0, 1.0, 0.0],
    [0.0, 2.0, 5.0, 1.0],
    [1.0, 0.0, 2.0, 3.0]
]]]).to(device)

# Feature map of frame 2 with single channel (the 9 shifted diagonally in this frame)
f_map_2 = torch.tensor([[[
    [1.0, 3.0, 0.0, 2.0],
    [4.0, 9.0, 1.0, 0.0],
    [0.0, 2.0, 5.0, 1.0],
    [1.0, 0.0, 2.0, 3.0]
]]]).to(device)

print(f"Feature map 1:\n{f_map_1.squeeze()}\n")
print(f"Feature map 2:\n{f_map_2.squeeze()}\n")
print(f"Shape of Feature map 1 (B, C, H, W): {f_map_1.shape}")
print(f"Shape of Feature map 2 (B, C, H, W): {f_map_2.shape}")

Feature map 1:
tensor([[9., 1., 0., 2.],
        [3., 4., 1., 0.],
        [0., 2., 5., 1.],
        [1., 0., 2., 3.]], device='cuda:0')

Feature map 2:
tensor([[1., 3., 0., 2.],
        [4., 9., 1., 0.],
        [0., 2., 5., 1.],
        [1., 0., 2., 3.]], device='cuda:0')

Shape of Feature map 1 (B, C, H, W): torch.Size([1, 1, 4, 4])
Shape of Feature map 2 (B, C, H, W): torch.Size([1, 1, 4, 4])


In [ ]:
# The pooling layer (non learnnable)

# Creating a pooling layer with 2x2 window and stride jump of 2
# Using the formula to calculate the width and height of the output tensor
# This object will automatically allocate memory of size calculated using that output formula
# W_out = floor((W_in - K) / S) + 1
# W_out: Output width, W_in = Input width (feature map), K = kernel width, S = Stride jump length
pooling_layer = nn.MaxPool2d(kernel_size=2, stride=2).to(device)

In [ ]:
# Execute forward pass on both the feature maps of the frames we defined above
output_1 = pooling_layer(f_map_1)
output_2 = pooling_layer(f_map_2)

print(f"Output 1 tensor shape: {output_1.shape}")
print(f"Output 1 tensor data:\n{output_1.squeeze()}\n")
print(f"Output 2 tensor shape: {output_2.shape}")
print(f"Output 2 tensor data:\n{output_2.squeeze()}")

# It it noticible that the output tensors are max pooled because both the output tensors are same unlike the input tensors
# It solved the Spatial hypersensitivity where a formula is too rigid and can break with a samll change
# Like the 9 in our example feature maps, moved to (1, 1) from (0, 0). In rigid matrix multiplication layer this is different
# We solve this using pooling that makes the operation translation invariant (different input but same output)
# It makes the output (input for the next layer) tensor smaller as well saving GPU space

Output 1 tensor shape: torch.Size([1, 1, 2, 2])
Output 1 tensor data:
tensor([[9., 2.],
        [2., 5.]], device='cuda:0')

Output 2 tensor shape: torch.Size([1, 1, 2, 2])
Output 2 tensor data:
tensor([[9., 2.],
        [2., 5.]], device='cuda:0')


In [ ]:
# FEW QUESTIONS TO ASK
# 1. Can you explain with detailed steps of how the autograd engine works for CNN? Show me the real thing.
# 2. What does shared parameter mean in CNN?
# 3. If doing convolution -> pool -> convolution -> pool multiple times and by layer 5 the feature map becomes 1 x 1, How does it even have the data for each and every feature in the image (eg. an edge, a corner or like a face)?



### Tracing a Complete CNN Cycle
Let's define the 3 layers and the initial values in VRAM.
*(Note: To keep the math clean and traceable, we will omit the bias $+b$ and focus entirely on the weight matrices, which are the core of the learning process).*

**1. The Input Image ($X$):** A $3 \times 3$ grayscale image (1 channel).
$$ X = \begin{bmatrix} 1 & 2 & 1 \\ 0 & 1 & 2 \\ 2 & 0 & 1 \end{bmatrix} $$

**2. Layer 1: Convolutional Layer ($L_1$)**
*   **Kernel ($W_1$):** A $2 \times 2$ weight matrix.
*   **Hyperparameters:** Stride = 1, Padding = 0.
$$ W_1 = \begin{bmatrix} 1 & 0 \\ -1 & 1 \end{bmatrix} $$

**3. Layer 2: Max Pooling Layer ($L_2$)**
*   **Hyperparameters:** $2 \times 2$ window, Stride = 2. (Compresses the feature map).

**4. Layer 3: Linear Layer ($L_3$)**
*   **Weight ($W_2$):** A single scalar weight (a $1 \times 1$ matrix) connecting the pooled feature to the final prediction. Let's initialize it to `2.0`.
*   **Target ($y$):** The ground truth label we want the network to predict is `10.0`.
*   **Learning Rate ($\alpha$):** `0.1`.

---

#### PHASE 1: THE FORWARD PASS (Building the DAG)

The Forward Pass is where data flows left-to-right. We execute the math and store the intermediate Feature Maps in VRAM because the Autograd engine needs them later.

##### Step 1.1: Layer 1 (Convolution Execution)
*   **The "How":** We slide the $2 \times 2$ Kernel ($W_1$) over the $3 \times 3$ Input Image ($X$). At each step, we perform an element-wise multiplication and sum the result (a dot product).
*   **The Math (Cross-Correlation):**
    *   **Top-Left Window:** $\begin{bmatrix} 1 & 2 \\ 0 & 1 \end{bmatrix} * \begin{bmatrix} 1 & 0 \\ -1 & 1 \end{bmatrix} = (1)(1) + (2)(0) + (0)(-1) + (1)(1) = 1 + 0 + 0 + 1 = \mathbf{2}$
    *   **Top-Right Window:** $\begin{bmatrix} 2 & 1 \\ 1 & 2 \end{bmatrix} * \begin{bmatrix} 1 & 0 \\ -1 & 1 \end{bmatrix} = (2)(1) + (1)(0) + (1)(-1) + (2)(1) = 2 + 0 - 1 + 2 = \mathbf{3}$
    *   **Bottom-Left Window:** $\begin{bmatrix} 0 & 1 \\ 2 & 0 \end{bmatrix} * \begin{bmatrix} 1 & 0 \\ -1 & 1 \end{bmatrix} = (0)(1) + (1)(0) + (2)(-1) + (0)(1) = 0 + 0 - 2 + 0 = \mathbf{-2}$
    *   **Bottom-Right Window:** $\begin{bmatrix} 1 & 2 \\ 0 & 1 \end{bmatrix} * \begin{bmatrix} 1 & 0 \\ -1 & 1 \end{bmatrix} = (1)(1) + (2)(0) + (0)(-1) + (1)(1) = 1 + 0 + 0 + 1 = \mathbf{2}$

*   **The Output (Feature Map 1):** We allocate a $2 \times 2$ tensor in RAM. Let's call it $F_1$.
$$ F_1 = \begin{bmatrix} 2 & 3 \\ -2 & 2 \end{bmatrix} $$

##### Step 1.2: Layer 2 (Max Pooling Execution)
*   **The "How":** We apply a $2 \times 2$ Max Pool to Feature Map 1 ($F_1$). Because the feature map is $2 \times 2$, this operation covers the entire map in a single step.
*   **The Math:** $\max(2, 3, -2, 2) = \mathbf{3}$
*   **The "Why" (The Argmax Cache):** During the forward pass, PyTorch doesn't just output the number `3`. It silently caches the *exact spatial index* where that `3` came from. Looking at $F_1$, the `3` was at the top-right corner, coordinate `(0, 1)`. The Autograd engine caches this coordinate so it knows exactly where to route the gradients later.
*   **The Output (Pooled Map):** We allocate a $1 \times 1$ tensor in RAM. Let's call it $P_1$.
$$ P_1 = \begin{bmatrix} 3 \end{bmatrix} $$

##### Step 1.3: Layer 3 (Linear / Fully Connected Execution)
*   **The "How":** We take the extracted pooled feature ($P_1$) and multiply it by our final Linear Weight ($W_2$) to get our prediction ($\hat{y}$).
*   **The Math:** $\hat{y} = P_1 \times W_2 = 3 \times 2.0 = \mathbf{6.0}$
*   **The Output (Prediction):** Our network predicts the value `6.0`.

##### Step 1.4: The Loss Calculation (The Final DAG Node)
*   **The "Why":** We need a single scalar number representing the mathematical error. We will use the standard **Mean Squared Error (MSE)**. To make the calculus derivative cleaner, computer scientists often multiply it by $\frac{1}{2}$. Formula: $L = \frac{1}{2}(\hat{y} - y)^2$
*   **The Math:**
    $$ L = \frac{1}{2}(6.0 - 10.0)^2 $$
    $$ L = \frac{1}{2}(-4.0)^2 $$
    $$ L = \frac{1}{2}(16.0) = \mathbf{8.0} $$
*   **The State of Memory:** We have a scalar `loss = 8.0`. The Forward Pass is complete. The DAG is fully built in RAM, pointing backward from $L \to \hat{y} \to P_1 \to F_1 \to X$.

---
#### PHASE 2: THE BACKWARD PASS (The Autograd Engine)

We currently have a `Loss` of `8.0`. We want to reduce it to `0.0`. To do this, we must traverse the DAG from right to left, using the Chain Rule to calculate how every weight contributed to that `8.0`.

**The Terminology:** As we move backward, the gradient calculated from the *subsequent* layer is passed backward to the *preceding* layer. We call this the **Upstream Gradient**. It acts as the mathematical error signal.

*(Note: We will use a Learning Rate of $\alpha = 0.01$ for this exercise to ensure we take a smooth step down the error surface without violently overshooting the target).*

##### Step 2.1: The Loss Derivative (The Starting Signal)
*   **The "Why":** We need to know the slope of the error at our exact prediction ($\hat{y} = 6.0$) relative to the target ($y = 10.0$).
*   **The Math:** The derivative of $L = \frac{1}{2}(\hat{y} - y)^2$ with respect to $\hat{y}$ is:
    $$ \frac{\partial L}{\partial \hat{y}} = (\hat{y} - y) = 6.0 - 10.0 = \mathbf{-4.0} $$
*   **The CS Meaning:** The negative sign tells the network: *"To reduce the error, you must increase the value of the prediction."* This `-4.0` is the very first **Upstream Gradient**, handed backward to Layer 3.

##### Step 2.2: Layer 3 Backward Pass (Linear Layer)
*   **The "Why":** Layer 3 has the equation $\hat{y} = P_1 \times W_2$. To satisfy the Chain Rule, the Autograd engine must calculate *two* distinct gradients here:
    1.  $\frac{\partial L}{\partial W_2}$: To know how to update the Linear Weight.
    2.  $\frac{\partial L}{\partial P_1}$: To pass the error signal backward to Layer 2.
*   **The Math (Gradient for the Weight):**
    $$ \frac{\partial L}{\partial W_2} = \text{Upstream Gradient} \times \text{Local Derivative (which is } P_1) $$
    $$ dW_2 = -4.0 \times 3.0 = \mathbf{-12.0} $$
    *(This is stored in VRAM as `W2.grad`)*
*   **The Math (Gradient for the Input):**
    $$ \frac{\partial L}{\partial P_1} = \text{Upstream Gradient} \times \text{Local Derivative (which is } W_2) $$
    $$ dP_1 = -4.0 \times 2.0 = \mathbf{-8.0} $$
*   **The State of Memory:** We hand this new Upstream Gradient ($dP_1 = -8.0$) backward to Layer 2.

##### Step 2.3: Layer 2 Backward Pass (Max Pooling)
*   **The "How":** Max Pooling has no learnable weights. Its only job during Backpropagation is to act as a **Router**.
*   **The "Why":** During the Forward Pass, the pooling window looked at Feature Map 1 ($F_1$) and selected the `3` at the top-right coordinate `(0, 1)`. The other three numbers (`2, -2, 2`) were deleted. Because they did not contribute to the final prediction, their local derivative is `0`. The network cannot blame them for the error.
*   **The Execution:** The GPU takes the Upstream Gradient ($-8.0$) and places it *exactly* back into the `(0, 1)` coordinate of a $2 \times 2$ matrix, filling the rest with zeros.
*   **The Output (Gradient of Feature Map 1):** Let's call this $dF_1$.
    $$ dF_1 = \begin{bmatrix} 0 & \mathbf{-8.0} \\ 0 & 0 \end{bmatrix} $$
*   **The State of Memory:** We hand this new $2 \times 2$ Upstream Gradient ($dF_1$) backward to Layer 1.

##### Step 2.4: Layer 1 Backward Pass (Convolution)
*   **The "How":** We must calculate the gradient for the Convolutional Kernel ($W_1$). As I explained previously, the GPU calculates this by taking the original Input Image ($X$) and performing a Cross-Correlation against the Upstream Gradient ($dF_1$).
*   **The Math ($\frac{\partial L}{\partial W_1}$):** We slide the $2 \times 2$ $dF_1$ matrix over the $3 \times 3$ Input Image ($X$).
    *   **Top-Left Weight Gradient:** $\begin{bmatrix} 1 & 2 \\ 0 & 1 \end{bmatrix} * \begin{bmatrix} 0 & -8 \\ 0 & 0 \end{bmatrix} = 1(0) + 2(-8) + 0(0) + 1(0) = \mathbf{-16.0}$
    *   **Top-Right Weight Gradient:** $\begin{bmatrix} 2 & 1 \\ 1 & 2 \end{bmatrix} * \begin{bmatrix} 0 & -8 \\ 0 & 0 \end{bmatrix} = 2(0) + 1(-8) + 1(0) + 2(0) = \mathbf{-8.0}$
    *   **Bottom-Left Weight Gradient:** $\begin{bmatrix} 0 & 1 \\ 2 & 0 \end{bmatrix} * \begin{bmatrix} 0 & -8 \\ 0 & 0 \end{bmatrix} = 0(0) + 1(-8) + 2(0) + 0(0) = \mathbf{-8.0}$
    *   **Bottom-Right Weight Gradient:** $\begin{bmatrix} 1 & 2 \\ 0 & 1 \end{bmatrix} * \begin{bmatrix} 0 & -8 \\ 0 & 0 \end{bmatrix} = 1(0) + 2(-8) + 0(0) + 1(0) = \mathbf{-16.0}$
*   **The Output (The Kernel Gradient):**
    $$ dW_1 = \begin{bmatrix} -16.0 & -8.0 \\ -8.0 & -16.0 \end{bmatrix} $$
    *(This is stored in VRAM as `W1.grad`)*

The Backward Pass is complete! The DAG is physically destroyed in RAM to free up memory.

---

#### PHASE 3: GRADIENT DESCENT (Memory Optimization)

We now execute the Optimizer. The Optimizer loops through RAM, finds `W1.grad` and `W2.grad`, and updates the original memory blocks using the formula: $W_{new} = W_{old} - (\alpha \times \nabla W)$. Our Learning Rate ($\alpha$) is `0.01`.

##### Step 3.1: Updating the Linear Weight ($W_2$)
*   **The Math:**
    $$ W_{2,new} = 2.0 - (0.01 \times -12.0) $$
    $$ W_{2,new} = 2.0 - (-0.12) = \mathbf{2.12} $$

##### Step 3.2: Updating the Convolutional Kernel ($W_1$)
*   **The Math:** We multiply the entire $dW_1$ matrix by `0.01` and subtract it from $W_1$.
    $$ W_{1,new} = \begin{bmatrix} 1 & 0 \\ -1 & 1 \end{bmatrix} - \left( 0.01 \times \begin{bmatrix} -16.0 & -8.0 \\ -8.0 & -16.0 \end{bmatrix} \right) $$
    $$ W_{1,new} = \begin{bmatrix} 1 & 0 \\ -1 & 1 \end{bmatrix} - \begin{bmatrix} -0.16 & -0.08 \\ -0.08 & -0.16 \end{bmatrix} $$
    $$ W_{1,new} = \begin{bmatrix} 1.16 & 0.08 \\ -0.92 & 1.16 \end{bmatrix} $$

---

#### PHASE 4: PROOF OF MINIMIZATION

As computer scientists, we don't assume the math worked. We prove it. Let's run a rapid Forward Pass using our *newly updated weights* to see if the Loss actually minimized.

1.  **New Feature Map 1 (Re-calculating the top-right Max Coordinate):**
    *   $\begin{bmatrix} 2 & 1 \\ 1 & 2 \end{bmatrix} * \begin{bmatrix} 1.16 & 0.08 \\ -0.92 & 1.16 \end{bmatrix} = (2 \times 1.16) + (1 \times 0.08) + (1 \times -0.92) + (2 \times 1.16)$
    *   $2.32 + 0.08 - 0.92 + 2.32 = \mathbf{3.8}$
2.  **New Pooled Map ($P_{1,new}$):** The new maximum signal extracted is `3.8`.
3.  **New Prediction ($\hat{y}_{new}$):** $P_{1,new} \times W_{2,new} = 3.8 \times 2.12 = \mathbf{8.056}$
4.  **New Total Loss:**
    $$ L_{new} = \frac{1}{2}(8.056 - 10.0)^2 $$
    $$ L_{new} = \frac{1}{2}(-1.944)^2 $$
    $$ L_{new} = \frac{1}{2}(3.779) = \mathbf{1.889} $$

#### The Ultimate Conclusion
*   **Starting Loss:** `8.0`
*   **Ending Loss:** `1.889`

By manually tracing the calculus of a sliding window, routing gradients through a Max Pool array, extracting the error of shared parameters, and updating the raw memory blocks, **you have mathematically proven that the Convolutional Neural Network has successfully learned.**

***